# H1 / H1b — Has the frequency of accusations of lying increased over time?

**H1**: the frequency of accusations of lying has increased over time.  
**H1b** (competing): it has remained stable.

Design: the unit is the **country-year accusation rate** = accusations ÷ sentences
spoken. Using the rate (not the raw count) controls for corpus coverage growing
over time. Estimation: (1) descriptive trends per country, (2) pooled trend with
country fixed effects, (3) robustness: dataset FE instead of country FE.

In [ ]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from lib import data, viz
viz.apply_style()

## Country-year rates

Denominator from the full corpus (149M rows, aggregated in DuckDB), numerator
from the accusation dataset.

In [ ]:
con = data.duck()   # excluded countries already filtered out of the views

cy = con.execute("""
WITH sents AS (
    SELECT country, CAST(substr(date,1,4) AS INT) AS year, COUNT(*) AS n_sentences
    FROM corpus WHERE date IS NOT NULL AND length(date) >= 4
    GROUP BY 1, 2
), accs AS (
    SELECT country, CAST(substr(date,1,4) AS INT) AS year, COUNT(*) AS n_accusations
    FROM accusations WHERE date IS NOT NULL AND length(date) >= 4
    GROUP BY 1, 2
)
SELECT s.country, s.year, s.n_sentences, COALESCE(a.n_accusations, 0) AS n_accusations
FROM sents s LEFT JOIN accs a USING (country, year)
ORDER BY country, year
""").df()

cy["rate"] = cy["n_accusations"] / cy["n_sentences"]          # per sentence
cy["rate_10k"] = cy["rate"] * 10_000                          # per 10k sentences
print(f"{cy['country'].nunique()} countries, years {cy['year'].min()}\u2013{cy['year'].max()}")
cy.head()

In [ ]:
# Guard against sparse country-years: rates from tiny denominators are noise.
MIN_SENTENCES = 10_000
cyf = cy[cy["n_sentences"] >= MIN_SENTENCES].copy()
print(f"kept {len(cyf):,} of {len(cy):,} country-years (>= {MIN_SENTENCES:,} sentences)")

## Descriptive: pooled rate over time

In [ ]:
pooled = (cyf.groupby("year")
             .apply(lambda g: pd.Series({
                 "rate_10k": g["n_accusations"].sum() / g["n_sentences"].sum() * 10_000,
                 "n_countries": g["country"].nunique()}), include_groups=False)
             .reset_index())

fig, ax = plt.subplots()
ax.plot(pooled["year"], pooled["rate_10k"], marker="o", ms=3)
ax2 = ax.twinx()
ax2.bar(pooled["year"], pooled["n_countries"], alpha=0.12, color="grey")
ax2.set_ylabel("countries in corpus (bars)")
ax.set_xlabel("year"); ax.set_ylabel("accusations per 10k sentences")
ax.set_title("Accusation rate over time (pooled)")
viz.savefig(fig, "h1_pooled_rate")
plt.show()
# NOTE: the country composition changes across years (bars). The FE models below
# handle this; the raw pooled line alone can mislead.

## Per-country trends

In [ ]:
countries = sorted(cyf["country"].unique())
ncol = 5; nrow = int(np.ceil(len(countries) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(16, 2.6 * nrow), sharex=True)
for ax, c in zip(axes.flat, countries):
    g = cyf[cyf["country"] == c]
    ax.plot(g["year"], g["rate_10k"], lw=1)
    ax.set_title(c, fontsize=9); ax.tick_params(labelsize=7)
for ax in axes.flat[len(countries):]:
    ax.axis("off")
fig.suptitle("Accusations per 10k sentences, by country", y=1.0)
fig.tight_layout()
viz.savefig(fig, "h1_country_trends")
plt.show()

## Trend test: Poisson with country FE and exposure offset

$\;n\_accusations_{ct} \sim \text{Poisson}$, offset $\log(n\_sentences_{ct})$,
country FE, linear year. The year coefficient is the average within-country
log-linear trend \u2014 exactly the H1 vs H1b test. Cluster-robust SEs by country.

In [ ]:
cyf["year_c"] = cyf["year"] - cyf["year"].mean()
m = smf.glm("n_accusations ~ year_c + C(country)", data=cyf,
            family=__import__("statsmodels.api", fromlist=["families"]).families.Poisson(),
            offset=np.log(cyf["n_sentences"]))\
       .fit(cov_type="cluster", cov_kwds={"groups": cyf["country"]})
b = m.params["year_c"]; ci = m.conf_int().loc["year_c"]
print(m.summary().tables[0])
print(f"\nyear coefficient: {b:.4f}  (95% CI {ci[0]:.4f} .. {ci[1]:.4f})")
print(f"=> {(np.exp(b)-1)*100:+.2f}% change in accusation rate per year, within country")

**Interpretation**: a positive, significant year coefficient supports **H1**;
a near-zero coefficient with a tight CI supports **H1b**.

TODO robustness (before the conference):
- balanced-panel check: restrict to countries observed \u2265 15 years;
- dataset FE instead of country FE (translation/source effects);
- decade splines instead of a linear year, to see *when* any rise happens.